# Parallel Dispatcher

> Notebook generated from [`examples/patterns/09_parallel_dispatcher.py`](../../examples/patterns/09_parallel_dispatcher.py).

| Field | Value |
|------|-------|
| **Dataset** | FEVER (embedded, 6 claims) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Fan-out with `asyncio.gather`; parallel results + aggregation; serial vs parallel timing comparison.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Parallel Dispatcher (Map-Reduce) — Parallel topic research
===================================================================
Pattern: SPEC / prismal.agents.patterns.parallel

Dataset: FEVER (Fact Extraction and VERification)
  • 185,445 claims about Wikipedia to verify.
  • Reference: https://huggingface.co/datasets/fever/fever
  • Why: The parallel dispatcher is perfect for verifying multiple
    independent claims in parallel (fan-out), then aggregating
    all results (fan-in). FEVER has exactly the kind of independent
    claims that can be distributed across workers.

Pattern description:
  make_parallel_dispatcher creates a LangGraph-compatible function:
  1. Fan-out: sends each task to worker_node via LangGraph Send()
  2. Fan-in: all workers finish and their results are aggregated
  3. Controlled by settings.parallel_max_workers (safety cap)
  4. Globally disableable with settings.parallel_enabled = False

Also demonstrates asyncio.gather directly for parallelism outside LangGraph.

Usage:
    uv run python examples/patterns/09_parallel_dispatcher.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import time
from typing import Any

from langchain_core.messages import HumanMessage, SystemMessage

from prismal.agents.patterns.parallel import make_parallel_dispatcher
from prismal.core.config import get_settings
from prismal.providers.registry import ProviderRegistry

## Dataset: FEVER claims for verification

In [ ]:
# Sample of FEVER claims (SUPPORTS / REFUTES / NOT ENOUGH INFO)
FEVER_CLAIMS = [
    {
        "id": "FC001",
        "claim": "Python was created by Guido van Rossum and first released in 1991.",
        "expected_label": "SUPPORTS",
        "topic": "python_history",
    },
    {
        "id": "FC002",
        "claim": "The Eiffel Tower is located in Berlin, Germany.",
        "expected_label": "REFUTES",
        "topic": "geography",
    },
    {
        "id": "FC003",
        "claim": "The GPT-4 model was developed by Google DeepMind.",
        "expected_label": "REFUTES",
        "topic": "ai_models",
    },
    {
        "id": "FC004",
        "claim": "LangChain is an open source framework for building applications with LLMs.",
        "expected_label": "SUPPORTS",
        "topic": "ai_frameworks",
    },
    {
        "id": "FC005",
        "claim": "The Rust language was designed with an emphasis on performance and memory safety.",
        "expected_label": "SUPPORTS",
        "topic": "programming_languages",
    },
    {
        "id": "FC006",
        "claim": "ChromaDB is a relational database designed for banking transactions.",
        "expected_label": "REFUTES",
        "topic": "vector_databases",
    },
]

## Worker: verification of a single claim

In [ ]:
async def verify_single_claim(claim_task: dict[str, Any]) -> dict[str, Any]:
    """Verify a single FEVER claim with the LLM.

    This is the worker that runs in parallel for each task.

    Args:
        claim_task: Dictionary with 'claim', 'id' and metadata.

    Returns:
        Verification result with label and justification.
    """
    settings = get_settings()
    llm = ProviderRegistry(settings=settings).get_llm()

    claim = claim_task["claim"]

    system_prompt = (
        "You are a fact checker. Analyze the given claim and "
        "classify it as:\n"
        "- SUPPORTS: the claim is true according to your knowledge\n"
        "- REFUTES: the claim is false according to your knowledge\n"
        "- NOT_ENOUGH_INFO: you do not have enough information\n\n"
        "Respond in the format:\n"
        "LABEL: [SUPPORTS|REFUTES|NOT_ENOUGH_INFO]\n"
        "JUSTIFICATION: [brief explanation]"
    )

    response = await llm.ainvoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Claim: {claim}"),
        ]
    )

    raw = str(response.content).strip()
    lines = raw.split("\n")

    # Parse response
    label = "NOT_ENOUGH_INFO"
    justification = raw

    for line in lines:
        if line.startswith("LABEL:"):
            label_part = line.replace("LABEL:", "").strip()
            if "SUPPORTS" in label_part:
                label = "SUPPORTS"
            elif "REFUTES" in label_part:
                label = "REFUTES"
            else:
                label = "NOT_ENOUGH_INFO"
        elif line.startswith("JUSTIFICATION:"):
            justification = line.replace("JUSTIFICATION:", "").strip()

    return {
        "claim_id": claim_task["id"],
        "claim": claim,
        "predicted_label": label,
        "expected_label": claim_task["expected_label"],
        "justification": justification,
        "correct": label == claim_task["expected_label"],
    }


## Parallelism demo with asyncio.gather

In [ ]:
async def run_parallel_verification(
    claims: list[dict],
    max_workers: int = 6,
) -> list[dict[str, Any]]:
    """Run verification in parallel using asyncio.gather.

    Demonstrates the fan-out/fan-in pattern without needing LangGraph.
    In a full LangGraph graph, you would use make_parallel_dispatcher.

    Args:
        claims: List of claims to verify.
        max_workers: Maximum concurrent workers.

    Returns:
        List of verification results.
    """
    # Limit concurrency with a semaphore
    semaphore = asyncio.Semaphore(max_workers)

    async def bounded_verify(claim: dict) -> dict[str, Any]:
        async with semaphore:
            return await verify_single_claim(claim)

    # Fan-out: all tasks in parallel (limited by semaphore)
    results = await asyncio.gather(
        *[bounded_verify(claim) for claim in claims],
        return_exceptions=True,
    )

    # Fan-in: filter errors and return successful results
    valid_results = []
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            valid_results.append(
                {
                    "claim_id": claims[i]["id"],
                    "error": str(result),
                    "correct": False,
                }
            )
        else:
            valid_results.append(result)

    return valid_results


## Demo of make_parallel_dispatcher

In [ ]:
def demo_langgraph_dispatcher() -> None:
    """Show how to use make_parallel_dispatcher in a LangGraph graph."""
    print("\n[make_parallel_dispatcher — Usage in LangGraph]")

    # In a real LangGraph graph, the dispatcher would be created like this:
    dispatcher = make_parallel_dispatcher(
        tasks_field="research_tasks",  # state field with the tasks
        worker_node="claim_verifier",  # worker node in the graph
        max_workers=6,  # concurrent worker cap
        on_empty="__end__",  # routing when there are no tasks
        task_key="_task",  # key to inject the task into the worker
    )

    print("  Dispatcher created with configuration:")
    print("    tasks_field : 'research_tasks'")
    print("    worker_node : 'claim_verifier'")
    print("    max_workers : 6")
    print()

    # Simulate graph state with pending tasks
    mock_state = {
        "research_tasks": [
            {"id": "T1", "query": "What is Python?"},
            {"id": "T2", "query": "What is LangChain?"},
            {"id": "T3", "query": "What is ChromaDB?"},
        ],
        "messages": [],
        "metadata": {},
    }

    # The dispatcher returns a list of Send() for LangGraph
    dispatch_result = dispatcher(mock_state)

    print(f"  State with {len(mock_state['research_tasks'])} tasks:")
    if isinstance(dispatch_result, list):
        print(f"  → {len(dispatch_result)} Send() emitted (parallel execution)")
        for send in dispatch_result[:3]:
            print(f"    • Send('{send.node}', task={send.arg.get('_task', {}).get('id', '?')})")
    else:
        print(f"  → Routing to: {dispatch_result}")

    # Empty state → routing to on_empty
    empty_state = {**mock_state, "research_tasks": []}
    empty_result = dispatcher(empty_state)
    print(f"\n  State with no tasks → routing to: {empty_result!r} (on_empty)")


## Main function

In [ ]:
async def main() -> None:
    print("=" * 70)
    print("  Parallel Dispatcher — Dataset: FEVER (Fact Verification)")
    print("=" * 70)

    # Show configuration
    settings = get_settings()
    max_w = getattr(settings, "parallel_max_workers", 10)
    parallel_enabled = getattr(settings, "parallel_enabled", True)
    print(f"\n  parallel_enabled    : {parallel_enabled}")
    print(f"  parallel_max_workers: {max_w}")
    print(f"  Claims to verify    : {len(FEVER_CLAIMS)}")

    # ── Benchmark: sequential vs parallel ─────────────────────────────────
    print("\n[Benchmark: Sequential vs Parallel]")

    # Parallel
    print(f"\n  Running {len(FEVER_CLAIMS)} verifications in parallel...")
    t0 = time.perf_counter()
    results_parallel = await run_parallel_verification(FEVER_CLAIMS, max_workers=6)
    t_parallel = time.perf_counter() - t0

    # Results
    print("\n  Verification results:")
    print(f"  {'ID':<8} {'Expected label':<20} {'Prediction':<20} {'OK?':>4}")
    print("  " + "─" * 55)
    for r in results_parallel:
        if "error" in r:
            print(f"  {r['claim_id']:<8} {'ERROR':<20} {str(r.get('error', ''))[:18]:<20} {'✗':>4}")
        else:
            ok = "✓" if r["correct"] else "✗"
            print(
                f"  {r['claim_id']:<8} {r['expected_label']:<20} {r['predicted_label']:<20} {ok:>4}"
            )

    # Metrics
    correct = sum(1 for r in results_parallel if r.get("correct", False))
    accuracy = correct / len(results_parallel) if results_parallel else 0
    errors = sum(1 for r in results_parallel if "error" in r)

    print(f"\n  Parallel time   : {t_parallel:.2f}s")
    print(f"  Claims          : {len(results_parallel)}")
    print(f"  Correct         : {correct}/{len(results_parallel)} ({accuracy:.1%})")
    if errors:
        print(f"  Errors          : {errors} (tolerated by asyncio.gather)")

    # Theoretical speedup
    assumed_seq_time = t_parallel * min(6, len(FEVER_CLAIMS))
    speedup = assumed_seq_time / t_parallel if t_parallel > 0 else 0
    print(f"  Estimated speedup: ~{speedup:.1f}x over sequential")

    # ── Demo of make_parallel_dispatcher ──────────────────────────
    demo_langgraph_dispatcher()

    # ── Sample justifications ─────────────────────────────────────
    print("\n[Sample justifications]")
    for r in results_parallel[:3]:
        if "justification" in r:
            print(f"  {r['claim_id']}: {r['justification'][:100]}...")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()